# Archivo .npz

Archivo contiene 624 etiquetas escalares (234 NORMAL, 390 PNEUMONIA). Para utilizarlo como Output data, debemos convertirlo a one-hot con forma (624,1,1,2) y guardarlo como .npz.

In [2]:
# ============================================================
# GENERAR DATASET NPZ DE 8 RADIOGRAFÍAS PARA ST EDGE AI
# 4 NORMAL + 4 PNEUMONIA
# ============================================================

from pathlib import Path

import cv2
import numpy as np
from skimage.color import rgb2gray
from skimage.transform import resize


# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------

RUTA_VAL = Path(
    r"C:\Users\Alonso\Jupyter\Tesis"
    r"\Test_Modelos_Cuantizacion"
    r"\Red_clasificacion_Imagenes"
    r"\Red_clasificacion_Radiografias"
    r"\chest_xray\val"
)

RUTA_NORMAL = RUTA_VAL / "NORMAL"
RUTA_PNEUMONIA = RUTA_VAL / "PNEUMONIA"

RUTA_SALIDA = (
    RUTA_VAL
    / "radiografias_val_8_st.npz"
)

MUESTRAS_POR_CLASE = 4

EXTENSIONES_VALIDAS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}


# ------------------------------------------------------------
# Buscar imágenes
# ------------------------------------------------------------

def obtener_imagenes(carpeta, cantidad):

    archivos = sorted(
        archivo
        for archivo in carpeta.iterdir()
        if (
            archivo.is_file()
            and archivo.suffix.lower()
            in EXTENSIONES_VALIDAS
        )
    )

    if len(archivos) < cantidad:
        raise RuntimeError(
            f"La carpeta {carpeta} contiene "
            f"{len(archivos)} imágenes válidas, "
            f"pero se necesitan {cantidad}."
        )

    return archivos[:cantidad]


archivos_normal = obtener_imagenes(
    RUTA_NORMAL,
    MUESTRAS_POR_CLASE,
)

archivos_pneumonia = obtener_imagenes(
    RUTA_PNEUMONIA,
    MUESTRAS_POR_CLASE,
)


# ------------------------------------------------------------
# Preprocesamiento idéntico al notebook
# ------------------------------------------------------------

def preparar_radiografia(ruta_imagen):

    imagen = cv2.imread(
        str(ruta_imagen)
    )

    if imagen is None:
        raise ValueError(
            f"No se pudo abrir la imagen: "
            f"{ruta_imagen}"
        )

    # Replica el preprocesamiento del entrenamiento:
    # resize devuelve valores aproximadamente en [0,1].
    imagen = resize(
        imagen,
        (150, 150, 3),
        mode="constant",
        anti_aliasing=True,
    )

    # Se conserva exactamente el procedimiento del notebook.
    imagen = rgb2gray(imagen)

    imagen = imagen.reshape(
        150,
        150,
        1,
    ).astype(np.float32)

    return imagen


# ------------------------------------------------------------
# Construir dataset equilibrado
# ------------------------------------------------------------

X_st = []
y_indices = []
archivos_utilizados = []

# Clase 0: NORMAL
for archivo in archivos_normal:

    X_st.append(
        preparar_radiografia(archivo)
    )

    y_indices.append(0)
    archivos_utilizados.append(
        ("NORMAL", archivo.name)
    )


# Clase 1: PNEUMONIA
for archivo in archivos_pneumonia:

    X_st.append(
        preparar_radiografia(archivo)
    )

    y_indices.append(1)
    archivos_utilizados.append(
        ("PNEUMONIA", archivo.name)
    )


X_st = np.asarray(
    X_st,
    dtype=np.float32,
)

y_indices = np.asarray(
    y_indices,
    dtype=np.int64,
)


# ------------------------------------------------------------
# Etiquetas one-hot requeridas por ST
# ------------------------------------------------------------

# NORMAL:
# 0 -> [1.0, 0.0]
#
# PNEUMONIA:
# 1 -> [0.0, 1.0]

y_onehot = np.eye(
    2,
    dtype=np.float32,
)[y_indices]

# (8,2) -> (8,1,1,2)
y_st = y_onehot.reshape(
    -1,
    1,
    1,
    2,
)


# ------------------------------------------------------------
# Validaciones
# ------------------------------------------------------------

assert X_st.shape == (
    8,
    150,
    150,
    1,
), X_st.shape

assert y_st.shape == (
    8,
    1,
    1,
    2,
), y_st.shape

assert X_st.dtype == np.float32
assert y_st.dtype == np.float32

assert np.all(
    (X_st >= 0.0)
    & (X_st <= 1.0)
), (
    "Las imágenes no están dentro del rango [0,1]."
)


# ------------------------------------------------------------
# Guardar un único NPZ
# ------------------------------------------------------------

np.savez_compressed(
    RUTA_SALIDA,
    x_test=X_st,
    y_test=y_st,
)


# ------------------------------------------------------------
# Verificación del archivo
# ------------------------------------------------------------

with np.load(
    RUTA_SALIDA,
    allow_pickle=False,
) as datos:

    X_verificacion = datos["x_test"]
    y_verificacion = datos["y_test"]

    print("\nClaves:", datos.files)
    print(
        "x_test:",
        X_verificacion.shape,
        X_verificacion.dtype,
    )
    print(
        "y_test:",
        y_verificacion.shape,
        y_verificacion.dtype,
    )

print("\nImágenes utilizadas:")

for indice, (
    clase,
    nombre,
) in enumerate(
    archivos_utilizados,
    start=1,
):
    print(
        f"{indice}. {clase}: {nombre}"
    )

print(
    "\nArchivo generado correctamente:"
)

print(RUTA_SALIDA)


Claves: ['x_test', 'y_test']
x_test: (8, 150, 150, 1) float32
y_test: (8, 1, 1, 2) float32

Imágenes utilizadas:
1. NORMAL: NORMAL2-IM-1427-0001.jpeg
2. NORMAL: NORMAL2-IM-1430-0001.jpeg
3. NORMAL: NORMAL2-IM-1431-0001.jpeg
4. NORMAL: NORMAL2-IM-1436-0001.jpeg
5. PNEUMONIA: person1946_bacteria_4874.jpeg
6. PNEUMONIA: person1946_bacteria_4875.jpeg
7. PNEUMONIA: person1947_bacteria_4876.jpeg
8. PNEUMONIA: person1949_bacteria_4880.jpeg

Archivo generado correctamente:
C:\Users\Alonso\Jupyter\Tesis\Test_Modelos_Cuantizacion\Red_clasificacion_Imagenes\Red_clasificacion_Radiografias\chest_xray\val\radiografias_val_8_st.npz
